# 03 — Notebook Proxy: Run a Chat App

The OpenShift AI notebook proxy lets you run a web app inside your notebook container and access it through a special URL — no separate deployment needed.

We will build a tiny [Chainlit](https://chainlit.io) chat app and access it at port 8080 via the proxy.

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
import os
os.environ["UV_EXTRA_INDEX_URL"] = "https://pypi.org/simple"

# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

## Step 1 — Install Chainlit

In [ ]:
!uv pip install chainlit openai -q

## Step 2 — Create the chat app

The `%%writefile` magic writes the cell content directly to a file.
The app reads the MaaS token from an environment variable set in Step 4, and streams the model's reply back token by token.

In [ ]:
%%writefile app.py
import os
import chainlit as cl
from openai import OpenAI

MAAS_URL = "https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/qwen36-27b/v1"

@cl.on_message
async def on_message(message: cl.Message):
    client = OpenAI(base_url=MAAS_URL, api_key=os.environ.get("MAAS_TOKEN", ""))
    stream = client.chat.completions.create(
        model="qwen36-27b",
        messages=[{"role": "user", "content": message.content}],
        stream=True
    )
    reply = cl.Message(content="")
    await reply.send()
    for chunk in stream:
        token = chunk.choices[0].delta.content or ""
        await reply.stream_token(token)
    await reply.update()

## Step 3 — Find your proxy URL

`NB_PREFIX` is set automatically by OpenShift AI and contains the path to your notebook (e.g. `/notebook/myproject/my-workbench`).
The proxy URL is that path with `/proxy/8080` appended.

In [ ]:
import os

NB_PREFIX = os.environ.get("NB_PREFIX", "")
ROOT_PATH = f"{NB_PREFIX}/proxy/8080"

print("Root path:", ROOT_PATH)
print()
print("Your proxy URL:")
print(f"  https://<your-notebook-host>{ROOT_PATH}")
print()
print("Replace <your-notebook-host> with the hostname from your browser address bar.")

## Step 4 — Start the app

The cell below starts Chainlit in the background on port 8080.
After running it, open the proxy URL from Step 3 in your browser.

In [ ]:
import base64
import subprocess
import time

# kill any existing chainlit process from a previous run
subprocess.run(["pkill", "-f", "chainlit run app.py"], capture_output=True)
time.sleep(1)

# get the MaaS token and pass it to the app process
result = subprocess.run(
    ["oc", "get", "secret", "maas-secret", "-o", "jsonpath={.data.token}"],
    capture_output=True, text=True
)
MAAS_TOKEN = base64.b64decode(result.stdout.strip()).decode()

env = os.environ.copy()
env["MAAS_TOKEN"] = MAAS_TOKEN

proc = subprocess.Popen(
    ["chainlit", "run", "app.py",
     "--port", "8080",
     "--host", "0.0.0.0",
     "--root-path", ROOT_PATH,
     "--no-reload"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(3)
print(f"Chainlit started (pid {proc.pid})")
print("Open the proxy URL from Step 3 in your browser to chat.")
print()
print("To stop the app, run: proc.terminate()")